# Final Project!

Author: Cameron Mullaney

This project is split into 2 major parts: 
1. Building a Linear Regression Model
2. Using the model to process streaming data

This will all be done using `pyspark` primarily, with pandas used as well. 

In [1]:
import pandas as pd
import numpy as np
import math
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

## Pyspark MLlib-specific tools for model evaluation
from pyspark.ml.regression import LinearRegression
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, VectorIndexer, OneHotEncoder, PCA
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator 

## Part I - Fitting the Model

Part I is comprised of 3 major tasks:

1. Creating a data transformation pipeline
2. Using that pipeline to fit a LR model
3. Evaluating the LR model.

Starting the spark session

In [3]:
spark = SparkSession.builder.appName('my_app').getOrCreate()

Reading the data in as a pandas DF, then converting to spark DF

In [5]:
power = pd.read_csv("power_ml_data.csv")
power = spark.createDataFrame(power) 
power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Looks good! Let's get to transforming

### Building the Pipeline

This will be a pretty complicated pipeline, containing many transformations.

For `hour`, I am casting it to a double, and then binarizing it with a cutoff of 6.5

In [6]:
hour_double = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_d FROM __THIS__"
)

In [7]:
hour_bin = Binarizer(threshold = 6.5, inputCol = "Hour_d", outputCol = "Hour_b")

For `month`, I am casting it to a double so I can One-hot encode it

In [8]:
month_double = SQLTransformer(
    statement="SELECT *, CAST(Month AS DOUBLE) AS Month_d FROM __THIS__"
)

In [9]:
month_ohe = OneHotEncoder(inputCol = "Month_d", outputCol = "Month_ohe")

Here I'm creating my `label` column

In [10]:
label_cast = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

Here I'm creating the list of variables I will send to the PCA

In [11]:
PCA_assembler = VectorAssembler(inputCols = ["Temperature", 
                                         "Humidity", 
                                         "Wind_Speed", 
                                         "General_Diffuse_Flows",
                                         "Diffuse_Flows"], 
                            outputCol = "pca_input", 
                            handleInvalid = 'keep')

In [12]:
pca = PCA(k = 2, inputCol = "pca_input", outputCol = "pca_features")

And here I'm running the PCA_assembler to fit the PCA.

In [13]:
temp = PCA_assembler.transform(
    label_cast.transform(power))

pca_fit = pca.fit(temp)        

This is the final assembler, creating the final "features" column for later fitting.

In [14]:
assembler = VectorAssembler(inputCols = ["pca_features", 
                                         "Hour_b", 
                                         "Power_Zone_1", 
                                         "Power_Zone_2",
                                         "Month_ohe"], 
                            outputCol = "features", 
                            handleInvalid = 'keep')

Time to test-run the transformations:

In [15]:
df = hour_double.transform(power)
df = hour_bin.transform(df)
df = month_double.transform(df)
df = month_ohe.fit(df).transform(df)
df = PCA_assembler.transform(df)
df = pca_fit.transform(df)
df = label_cast.transform(df)
temp = assembler.transform(df)

In [16]:
temp.show(5, truncate = False)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+------------------------------+----------------------------------------+-----------+-------------------------------------------------------------------------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_d|Hour_b|Month_d|Month_ohe     |pca_input                     |pca_features                            |label      |features                                                                             |
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+------------------------------+----------------------------------------+-----------+-------------------------------------------------------------------------------------+
|6.559      |73.8  

This looks good! Time to create our Pipeline

First I'll define what kind of model we want

In [17]:
lr = LinearRegression(featuresCol = "features", labelCol = "label")

In [18]:
pipe_1 = Pipeline(stages = [hour_double, hour_bin, month_double, month_ohe, PCA_assembler, pca, label_cast, assembler, lr])

### Building and fitting the model

Creating our list of potential parameters

In [19]:
LR_paramGrid = ParamGridBuilder()\
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])\
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])\
    .build()


Setting up our Cross validator to find our optimal parameters

In [21]:
LRCV = CrossValidator(estimator = pipe_1,
                      estimatorParamMaps = LR_paramGrid,
                      evaluator = RegressionEvaluator(metricName = "rmse"),
                      numFolds=5,
                      parallelism = 4,
                      seed = 12)

This is going to take a while - and produce many warnings.

In [23]:
LRCV_model = LRCV.fit(power)

26/04/30 15:44:03 WARN Instrumentation: [8297042f] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 15:44:03 WARN Instrumentation: [9bc37037] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 15:44:03 WARN Instrumentation: [c45fe3df] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 15:44:03 WARN Instrumentation: [6da758a4] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 15:44:03 WARN Instrumentation: [8297042f] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 15:44:04 WARN Instrumentation: [9bc37037] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 15:44:04 WARN Instrumentation: [c45fe3df] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 15:44:04 WARN Instrumentation: [6da758a4] Cholesky solv

### Reporting model values

Okay! Let's see what our optimal values are

In [24]:
LR_list = []
for i in range(len(LR_paramGrid)):
    LR_list.append([LRCV_model.avgMetrics[i], LR_paramGrid[i].values()])
LR_list.sort(key=lambda x: x[0])
print("CV Error \t\t\t [regParam, ElasticParam]")
LR_list[0]

CV Error 			 [regParam, ElasticParam]


[2147.879982678048, dict_values([0.1, 0.1])]

Training RMSE:

In [25]:
LR_pred = LRCV_model.transform(power)
training_rmse = RegressionEvaluator(metricName = "rmse").evaluate(LR_pred)
print("Training RMSE: \t", training_rmse)

Training RMSE: 	 2147.0973203215863


Residuals:

In [26]:
resid_df = LR_pred.withColumn("residual", col("label") - col("prediction"))
resid_df.select("residual", "label", "prediction").show()

+------------------+-----------+------------------+
|          residual|      label|        prediction|
+------------------+-----------+------------------+
|-638.6844049043611|20240.96386| 20879.64826490436|
| 1471.136738545676|20131.08434|18659.947601454325|
| 1463.958504022663|19668.43373|18204.475225977338|
|1308.8696312354987|18899.27711|  17590.4074787645|
|1445.3432101497056|18442.40964|16997.066429850296|
|1612.6517153933491|18130.12048|16517.468764606652|
|1852.0122508872864|17945.06024|16093.047989112712|
|1736.7707088120023|17459.27711|15722.506401187997|
|1754.6696976219519|17025.54217|15270.872472378049|
|1856.0333437770878|16794.21687|14938.183526222912|
|1985.7932322312754|16638.07229|14652.279057768725|
| 1980.376702661104|16395.18072|14414.804017338896|
|2034.8443820286557|16117.59036|14082.745977971344|
|2197.8897606053015| 15822.6506|  13624.7608393947|
|2222.0367874452804|15672.28916| 13450.25237255472|
|2294.9068209740126|15597.10843|13302.201609025988|
| 2355.57556

## Streaming

Part II is split into two sections - one of which is in "Finalproj_stream.py".

In this notebook, I set up my query and data stream, which will look in the "power_stream" folder for csv's. Then, I'll read in those csv's transform them using `LRCV_model` from part I, and then produce predictions and residuals.

Next, from the same stream, I'll rename "Power_Zone_3" to "label", and join these two tables on this "label" column, and print the result in the console.

In [27]:
powerStream = spark.readStream.csv("power_stream", schema = power.schema, header = True)

Prediction table

In [28]:
stream_predictions = LRCV_model.transform(powerStream) \
    .withColumn("residual", col("label") - col("prediction")) \
    .select("label", "prediction", "residual")

Everything else table (with renamed response column)

In [29]:
stream_label = powerStream.withColumnRenamed("Power_Zone_3", "label")

Now we join them! This one will be used in the final query.

In [30]:
joined_stream = stream_predictions.join(stream_label, on="label")

Great! Now to write the .py file so I can test this!

Now that I've got these ready to go, it's time to start the query, and then run the .py script loop!

In [38]:
query = joined_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/30 15:56:34 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-705bd0d6-d3f0-47a0-be4f-4eaf37a31839. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 15:56:34 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19077.44681|22289.885689308285|-3212.4388793082835|       25.2|   60.12|     4.925|                13.15|        10.96| 46899.25602| 27765.56017|   10|  18|
|26007.09677|25602.930378834542| 404.16639116545775|      10.75|   66.33|     0.085|                0.066|        0.119| 43739.23404|  26769.5122|    3|  20|
|21251.39939| 24827.96365926825|-3576.5642692682522|      22.36|   67.79|     4.933|                3.564|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|6818.313253| 5282.016706123653| 1536.2965468763468|      14.66|    81.3|     0.073|                0.066|        0.104| 18523.07692| 12269.00826|   11|   7|
|26534.56067| 27124.91890017424| -590.3582301742426|      25.28|   46.28|     4.908|                0.117|        0.104| 31038.93688| 21687.34177|    7|   2|
|27397.81818|25935.110153782174| 1462.7080262178242|      18.42|    70.7|      0.08|                33.12|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15720.72727|15686.764966748915|  33.96230325108445|      12.14|    71.8|     0.082|                0.018|        0.167| 24106.43703| 13391.85336|    4|   2|
|14103.87097| 14155.92787863087| -52.05690863086966|       14.9|    84.2|     0.078|                0.026|        0.211| 24204.25532|  13990.2439|    3|   5|
|13732.25806|13536.346626260152| 195.91143373984778|      12.43|    73.5|     0.085|                0.044|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13231.80723|12829.840472046862| 401.96675795313786|      3.706|    79.7|     0.085|                0.099|        0.134| 23805.56962| 15348.32827|    1|   7|
|25402.18182| 22732.27397691216|   2669.90784308784|      17.23|    80.8|     0.085|                23.06|        19.87| 37536.10334| 20606.51731|    4|  18|
|17065.16129| 18257.20363704455|-1192.0423470445494|      13.72|    76.8|     0.088|                205.5|       

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19416.67712|21162.101156569443|-1745.4240365694423|      20.74|    77.2|     4.912|                0.095|        0.078| 28499.53385| 18665.25871|    8|   3|
|27972.92308|27652.009510199074|  320.9135698009268|      21.41|   47.91|     0.081|                24.56|        24.08| 44859.33775|  27598.7526|    6|  20|
|12389.54407| 11698.91892869938|  690.6251413006194|      20.78|    81.3|     0.082|                183.1|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14232.28916|13681.534370989548| 550.7547890104524|      3.541|    80.8|     0.085|                0.055|        0.115|  22584.3038| 13798.17629|    1|   5|
| 10328.6747| 9465.548007661257| 863.1266923387429|      11.11|   69.24|     4.917|                0.022|         0.13| 21575.38462| 17642.97521|   11|   4|
|8557.022809| 7326.603815407345|1230.4189935926552|       11.0|    78.5|     0.081|                257.2|        30.2

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16158.07229|12096.090154029267|  4061.982135970733|      19.91|   68.05|     0.073|                390.0|        249.5| 30412.30769| 23243.80165|   11|  11|
|12214.46809|15414.649400982393|-3200.1813109823925|      22.91|    83.2|     0.083|                316.7|        219.2| 38568.05252| 22451.45228|   10|  13|
| 11572.5228| 8560.615717765011| 3011.9070822349895|      17.43|    70.0|      0.08|                0.029|      

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|24934.73684| 24161.55386134427|  773.1829786557319|      19.64|   62.65|     0.079|                0.051|        0.093| 41585.31148| 25066.25387|    5|  22|
|25853.42714|25259.969746502124|  593.4573934978762|       13.2|   54.27|     4.919|                0.066|        0.082| 42901.01695| 25309.42249|    2|  20|
|27235.10972|26442.521581914472|   792.588138085528|      25.65|    75.9|     4.945|                135.5|      

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12312.87449|14356.729035907922|-2043.8545459079214|      16.77|    88.4|     4.914|                150.5|        123.5| 27811.67213| 17791.95046|    5|   8|
|31312.46862|29908.678853379693| 1403.7897666203062|      33.29|   32.12|     4.913|                645.9|        67.36| 40345.51495| 26418.98734|    7|  16|
|16782.15075|13476.435062141949| 3305.7156878580518|      11.31|   42.64|     0.086|                594.2|      

In [39]:
query.stop()

26/04/30 16:02:05 WARN DAGScheduler: Failed to cancel job group c5ce477d-7829-4bee-9aa7-33e59b92b27f. Cannot find active jobs for it.
26/04/30 16:02:06 WARN DAGScheduler: Failed to cancel job group c5ce477d-7829-4bee-9aa7-33e59b92b27f. Cannot find active jobs for it.
